<a href="https://colab.research.google.com/github/takumi-maker/dmd/blob/main/bond_PCA_arbitrage_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

class TreasuryPCAStrategy:
    def __init__(self, data, formation_window=20, trading_window=15, entry_z=2.0, exit_z=0.0):
        """
        論文の第3.3-3.4節に基づく、PCAを用いた統計的裁定取引

        :param data: 4銘柄（2Y, 5Y, 10Y, 30Y）の価格データ (DataFrame)
        """
        self.data = data
        self.formation_window = formation_window
        self.trading_window = trading_window
        self.entry_z = entry_z
        self.exit_z = exit_z

    def get_pca_weights(self, train_data):
        """
        論文のロジック:
        1. 対数価格の階差（リターン）にPCAを適用
        2. 最小の固有値に対応する固有ベクトル（第4主成分）を共和分ベクトルとして採用
        """
        # 対数価格の階差を計算 (第3.3節の記述より)
        rate = train_data
        #returns = rate.diff().dropna()
        returns = rate


        # PCAの実行 (全4コンポーネント)
        pca = PCA(n_components=2)
        pca.fit(returns)

        # 論文の第3.4節より:
        # 上位3つの要因(レベル, スロープ, カーブ)に直交する「最後の主成分」を取得
        # pca.components_ の最後の行が最小分散のベクトル
        weights = pca.components_[-1]

        # 最初の資産のウェイトを1に正規化
        weights = weights / weights[0]
        return weights

    def backtest(self):

        bars_per_day = 1
        formation_bars = self.formation_window * bars_per_day
        trading_bars = self.trading_window * bars_per_day

        total_bars = len(self.data)
        results = []
        continue_flag = False

        # ロール・フォワード
        for start_idx in range(formation_bars, total_bars  , trading_bars):
            # --- 形成期間 (Formation Period) ---
            train_start = start_idx - formation_bars
            train_data = self.data.iloc[train_start:start_idx]

            # PCAによるウェイト推定
            weights = self.get_pca_weights(train_data)
            weights = np.round(weights, 1)
            print("weights",weights)
            for j in range(len(weights)):
              if abs(weights[j])>20:
                continue_flag = True

            if continue_flag:
              continue

            # 複利ベースでのスプレッド（合成ポートフォリオ）
            train = train_data
            spread_train = np.dot(train.values, weights)
            #print("train",spread_train)
            mu = np.nanmean(spread_train)
            sigma = np.nanstd(spread_train)

            # --- 運用期間 (Trading Period) ---
            test_start = start_idx
            test_end = start_idx + trading_bars
            test_data = self.data.iloc[test_start:test_end]

            log_test = test_data
            spread_test = np.dot(log_test.values, weights)
            z_scores = (spread_test - mu) / sigma

            if test_start <= total_bars < test_end:
              print("current_z_score:",z_scores[-1])
              print("weights:",weights)

            # ポジション管理 (前回と同様)
            position = 0 # 1: Long, -1: Short
            entry_spread = 0

            if len(z_scores)==trading_bars:
              print("close position!")

            for i in range(len(z_scores)):
                z = z_scores[i]
                curr_spread = spread_test[i]
                curr_time = test_data.index[i]
                if test_start <= total_bars < test_end:
                    print("position:",position)

                if i == len(z_scores) - 1:
                  if test_start <= total_bars < test_end:
                    print("current_position:",position)

                if position == 0:
                    if z <= -self.entry_z: # 割安 -> 買い
                        position = 1
                        entry_spread = curr_spread
                        results.append({'Time': curr_time, 'Action': 'BUY', 'Price': curr_spread, 'Z': z})
                        if i == len(z_scores) - 1:
                          if test_start <= total_bars < test_end:
                            print("bet widening")
                    elif z >= self.entry_z: # 割高 -> 売り
                        position = -1
                        entry_spread = curr_spread
                        results.append({'Time': curr_time, 'Action': 'SELL', 'Price': curr_spread, 'Z': z})
                        if i == len(z_scores) - 1:
                          if test_start <= total_bars < test_end:
                            print("bet tightning")

                elif position == 1 and z >= -self.exit_z: # 利益確定
                    results.append({'Time': curr_time, 'Action': 'EXIT', 'Price': curr_spread, 'Profit': curr_spread - entry_spread})
                    position = 0
                elif position == -1 and z <= self.exit_z: # 利益確定
                    results.append({'Time': curr_time, 'Action': 'EXIT', 'Price': curr_spread, 'Profit': entry_spread - curr_spread})
                    position = 0
                if i == len(z_scores) - 1 and position == 1:
                  results.append({'Time': curr_time, 'Action': 'EXIT', 'Price': curr_spread, 'Profit': curr_spread - entry_spread})
                if i == len(z_scores) - 1 and position == -1:
                  results.append({'Time': curr_time, 'Action': 'EXIT', 'Price': curr_spread, 'Profit': entry_spread - curr_spread})

        return pd.DataFrame(results)

In [ ]:
# --- テスト実行用データ作成 ---

    # サンプルデータの生成
n = 390 * 60
idx = pd.date_range("2015-01-01", periods=n, freq="T")
# 共通要因(ランダムウォーク)
common = np.cumsum(np.random.normal(0, 0.01, n))
df_1 = pd.read_csv("債券複利データセット3.csv")
df_1 = df_1.drop(df_1.columns[0], axis=1)
columns_len = len(df_1.columns)
length = len(df_1)
df_1
y_df = df_1
#y_df = y_df.drop('GJGC7',axis=1)
y_df = y_df.drop('GJGC2',axis=1)
y_df = y_df.drop('GJGC5',axis=1)
#y_df = y_df.drop('GJGC10',axis=1)
y_df = y_df.drop('GJGC15',axis=1)
y_df = y_df.drop('GJGC20',axis=1)
y_df = y_df.drop('GJGC30',axis=1)
y_df = y_df.drop('GJGC40',axis=1)

# 実行
model = TreasuryPCAStrategy(y_df)
trades = model.backtest()

if not trades.empty:
    print(f"総トレード数: {len(trades[trades['Action']=='EXIT'])}")
    print(f"累積損益: {trades['Profit'].sum():.6f}")
    print(trades)

weights [ 1. -1.]
close position!
weights [ 1.  -0.9]
close position!
weights [ 1.  -0.9]
close position!
weights [ 1.  -0.9]
close position!
weights [ 1.  -0.8]
close position!
weights [ 1.  -1.1]
close position!
weights [ 1. -1.]
close position!
weights [ 1.  -1.7]
close position!
weights [ 1. -1.]
close position!
weights [ 1.  -1.4]
close position!
weights [ 1.  -1.4]
close position!
weights [ 1.  -1.1]
close position!
weights [ 1.  -1.2]
close position!
weights [ 1.  -1.4]
close position!
weights [ 1.  -1.4]
close position!
weights [ 1.  -1.2]
close position!
weights [ 1.  -0.8]
close position!
weights [ 1.  -1.3]
close position!
weights [ 1.  -1.3]
close position!
weights [ 1.  -1.9]
close position!
weights [ 1.  -1.6]
close position!
weights [ 1.  -2.5]
close position!
weights [ 1.  -6.3]
close position!
weights [ 1.  -8.5]
close position!
weights [ 1.  -2.3]
close position!
weights [ 1.  -1.5]
close position!
weights [ 1.  -1.8]
close position!
weights [ 1.  -7.4]
close position

/tmp/ipykernel_1206/3638330553.py:5: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  idx = pd.date_range("2015-01-01", periods=n, freq="T")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

class TreasuryPCAStrategy:
    def __init__(self, data, formation_window=20, trading_window=15, entry_z=2.0, exit_z=0.0):
        """
        論文の第3.3-3.4節に基づく、PCAを用いた統計的裁定取引

        :param data: 4銘柄（2Y, 5Y, 10Y, 30Y）の価格データ (DataFrame)
        """
        self.data = data
        self.formation_window = formation_window
        self.trading_window = trading_window
        self.entry_z = entry_z
        self.exit_z = exit_z

    def get_pca_weights(self, train_data):
        """
        論文のロジック:
        1. 対数価格の階差（リターン）にPCAを適用
        2. 最小の固有値に対応する固有ベクトル（第4主成分）を共和分ベクトルとして採用
        """
        # 対数価格の階差を計算 (第3.3節の記述より)
        rate = train_data
        #returns = rate.diff().dropna()
        returns = rate


        # PCAの実行 (全4コンポーネント)
        pca = PCA(n_components=2)
        pca.fit(returns)

        # 論文の第3.4節より:
        # 上位3つの要因(レベル, スロープ, カーブ)に直交する「最後の主成分」を取得
        # pca.components_ の最後の行が最小分散のベクトル
        weights = pca.components_[-1]

        # 最初の資産のウェイトを1に正規化
        weights = weights / weights[0]
        return weights

    def predict(self):
      total_bars = len(self.data)

    def backtest_and_predict(self):

        bars_per_day = 1
        formation_bars = self.formation_window * bars_per_day
        trading_bars = self.trading_window * bars_per_day

        total_bars = len(self.data)
        results = []
        continue_flag = False

        # ロール・フォワード
        for start_idx in range(formation_bars, total_bars  - trading_bars, trading_bars):
            if start_idx < total_bars <
            # --- 形成期間 (Formation Period) ---
            train_start = start_idx - formation_bars
            train_data = self.data.iloc[train_start:start_idx]

            # PCAによるウェイト推定
            weights = self.get_pca_weights(train_data)
            print("weights",weights)
            for j in range(len(weights)):
              if abs(weights[j])>30:
                continue_flag = True

            if continue_flag:
              continue

            # 複利ベースでのスプレッド（合成ポートフォリオ）
            train = train_data
            spread_train = np.dot(train.values, weights)
            #print("train",spread_train)
            mu = np.nanmean(spread_train)
            sigma = np.nanstd(spread_train)



            # --- 運用期間 (Trading Period) ---
            test_start = start_idx
            test_end = train_end + trading_bars
            test_data = self.data.iloc[test_start:test_end]

            log_test = test_data
            spread_test = np.dot(log_test.values, weights)
            z_scores = (spread_test - mu) / sigma

            # ポジション管理 (前回と同様)
            position = 0 # 1: Long, -1: Short
            entry_spread = 0

            for i in range(len(z_scores)):
                z = z_scores[i]
                curr_spread = spread_test[i]
                curr_time = test_data.index[i]

                if position == 0:
                    if z <= -self.entry_z: # 割安 -> 買い
                        position = 1
                        entry_spread = curr_spread
                        results.append({'Time': curr_time, 'Action': 'BUY', 'Price': curr_spread, 'Z': z})
                    elif z >= self.entry_z: # 割高 -> 売り
                        position = -1
                        entry_spread = curr_spread
                        results.append({'Time': curr_time, 'Action': 'SELL', 'Price': curr_spread, 'Z': z})

                elif position == 1 and z >= -self.exit_z: # 利益確定
                    results.append({'Time': curr_time, 'Action': 'EXIT', 'Price': curr_spread, 'Profit': curr_spread - entry_spread})
                    position = 0
                elif position == -1 and z <= self.exit_z: # 利益確定
                    results.append({'Time': curr_time, 'Action': 'EXIT', 'Price': curr_spread, 'Profit': entry_spread - curr_spread})
                    position = 0
                if i == len(z_scores) - 1 and position == 1:
                  results.append({'Time': curr_time, 'Action': 'EXIT', 'Price': curr_spread, 'Profit': curr_spread - entry_spread})
                if i == len(z_scores) - 1 and position == -1:
                  results.append({'Time': curr_time, 'Action': 'EXIT', 'Price': curr_spread, 'Profit': entry_spread - curr_spread})

        return pd.DataFrame(results)



In [ ]:
print(trades)

     Time Action     Price         Z    Profit
0      25   SELL  0.255262  2.475011       NaN
1      31   EXIT  0.138975       NaN  0.116287
2      35    BUY -0.382371 -2.655285       NaN
3      40   EXIT -0.322478       NaN  0.059893
4      47   SELL -0.278879  3.821056       NaN
..    ...    ...       ...       ...       ...
211  1069   EXIT  0.496293       NaN -0.008138
212  1074   SELL  2.158635  2.953585       NaN
213  1076   EXIT  1.637356       NaN  0.521279
214  1096    BUY -0.003436 -2.621213       NaN
215  1099   EXIT  0.001303       NaN  0.004739

[216 rows x 5 columns]
